# V7.1 — YOLO26m fine-tune trên v2 reviewed 181126 (RTX 4060)

Fine-tune từ checkpoint seed85 curated trên `finding_nemo_human_reviewed_6cls_v2_171126` sau human/model review ngày 181126. Train có 20,949 ảnh; val/test giữ nguyên. Checkpoint được lưu local và đồng bộ Camber mỗi 5 epoch.

In [1]:
# Cấu hình chính
from pathlib import Path
import os, shutil, subprocess, sys, json, time

RUN_NAME = 'yolo26m_finetune_v2_reviewed_181126'
EXPERIMENT = 'finding_v7p1_171126'
REMOTE_ROOT = f'stash://bunpmc/projects/{EXPERIMENT}'
REMOTE_RUN = f'{REMOTE_ROOT}/{RUN_NAME}'
SOURCE_WEIGHTS = 'stash://bunpmc/projects/finding_seed/seed85/curated_yolo26m/weights/best.pt'

EPOCHS = 30          # 30 epoch mới, tính từ seed85 best
IMGSZ = 640
BATCH = 8           # RTX 4060 Laptop 8 GB: ưu tiên ổn định WDDM/CUDA
SEED = 85
SYNC_EVERY = 5
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def find_repo():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'benchmakring_transmission_line_defect_model').is_dir():
            return p
    raise RuntimeError('Hãy mở notebook từ repo C:/Users/intern/Documents/SEP490/train')

ROOT = find_repo()
os.chdir(ROOT)  # checkpoint CLI lưu một số path tương đối; resume luôn từ repo root
DATA_ROOT = ROOT / 'benchmakring_transmission_line_defect_model/data/finding_nemo_human_reviewed_6cls_v2_171126'
DATA_YAML = DATA_ROOT / 'data_local.yaml'
assert (DATA_ROOT / 'REVIEW_APPLIED_181126.json').is_file(), 'Dataset chưa áp review 181126'
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
EXPECTED = {'train': 20949, 'val': 2271, 'test': 1931}
for split in ('train', 'val', 'test'):
    for kind in ('images', 'labels'):
        folder = DATA_ROOT / kind / split
        assert folder.is_dir() and any(folder.iterdir()), f'Dataset thiếu {folder}'
    image_count = sum(p.suffix.lower() in IMAGE_EXTS for p in (DATA_ROOT / 'images' / split).iterdir())
    label_count = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    assert image_count == label_count == EXPECTED[split], (split, image_count, label_count)
DATA_YAML.write_text(
    'path: ' + DATA_ROOT.resolve().as_posix() + '\n'
    'train: images/train\nval: images/val\ntest: images/test\n'
    'names:\n  0: insulator\n  1: insulator_broken\n  2: insulator_flashover\n'
    '  3: bird_nest\n  4: foreign_object\n  5: broken_strand\n',
    encoding='utf-8')
PROJECT = ROOT / 'runs' / EXPERIMENT
RUN_DIR = PROJECT / RUN_NAME
INIT_WEIGHT = ROOT / 'weights' / 'v7p1_seed85_curated_yolo26m_best.pt'
PROJECT.mkdir(parents=True, exist_ok=True)
INIT_WEIGHT.parent.mkdir(parents=True, exist_ok=True)
assert DATA_YAML.is_file(), DATA_YAML
print('ROOT       :', ROOT)
print('DATA       :', DATA_YAML)
print('LOCAL RUN  :', RUN_DIR)
print('CAMBER RUN :', REMOTE_RUN)

ROOT       : D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection
DATA       : D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_human_reviewed_6cls_v2_171126\data_local.yaml
LOCAL RUN  : D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p1_171126\yolo26m_finetune_v2_reviewed_181126
CAMBER RUN : stash://bunpmc/projects/finding_v7p1_171126/yolo26m_finetune_v2_reviewed_181126


## 1. Môi trường
Nếu import Ultralytics lỗi, bỏ comment dòng `%pip` rồi chạy cell và restart kernel. Giữ phiên bản cố định để lần resume không đổi môi trường.

In [2]:
# %pip install -U "ultralytics==8.4.172" python-dotenv
import torch, ultralytics
from ultralytics import YOLO
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass
assert torch.cuda.is_available(), 'Không thấy CUDA. Kiểm tra PyTorch CUDA/driver NVIDIA.'
print('Ultralytics:', ultralytics.__version__)
print('PyTorch    :', torch.__version__)
print('GPU        :', torch.cuda.get_device_name(0))
print('VRAM       :', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

Ultralytics: 8.4.172
PyTorch    : 2.14.1+cu126
GPU        : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM       : 8.0 GB


## 2. Kết nối Camber và tải checkpoint
Cần `CAMBER_API_KEY` trong `.env` hoặc Camber CLI đã đăng nhập. Cell không in API key.

In [3]:
def load_camber_key():
    if os.environ.get('CAMBER_API_KEY'):
        return
    env_file = ROOT / '.env'
    if env_file.is_file():
        for line in env_file.read_text(encoding='utf-8').splitlines():
            key, sep, value = line.partition('=')
            if sep and key.strip() == 'CAMBER_API_KEY' and value.strip():
                os.environ['CAMBER_API_KEY'] = value.strip().strip(chr(34)).strip(chr(39)); return
    raise RuntimeError('Thiếu CAMBER_API_KEY trong environment hoặc .env của repo')

def wsl_path(path: Path):
    path = path.resolve()
    return f'/mnt/{path.drive.rstrip(chr(58)).lower()}{path.as_posix().split(chr(58),1)[1]}'

CAMBER = '/home/dotru/.camber/bin/camber'
load_camber_key()
CAMBER_ENV = os.environ.copy()
wslenv = [x for x in CAMBER_ENV.get('WSLENV','').split(':') if x]
if 'CAMBER_API_KEY' not in wslenv: wslenv.append('CAMBER_API_KEY')
CAMBER_ENV['WSLENV'] = ':'.join(wslenv)

def camber(*parts, check=True):
    converted = [wsl_path(x) if isinstance(x, Path) else str(x) for x in parts]
    command = ['wsl','--',CAMBER,'stash',*converted]
    p = subprocess.run(command,env=CAMBER_ENV,capture_output=True,text=True)
    if check and p.returncode:
        raise RuntimeError((p.stdout or p.stderr)[-3000:])
    return p

auth = subprocess.run(['wsl','--',CAMBER,'me'],env=CAMBER_ENV,capture_output=True,text=True)
assert auth.returncode == 0, (auth.stdout or auth.stderr)[-1200:]
print('Camber authentication OK via WSL:', CAMBER)

def ensure_remote_dir(uri):
    if camber('ls',uri,check=False).returncode != 0:
        made = camber('mkdir',uri,check=False)
        if made.returncode != 0 and camber('ls',uri,check=False).returncode != 0:
            raise RuntimeError((made.stdout or made.stderr)[-1200:])

for uri in ('stash://bunpmc/projects',REMOTE_ROOT,REMOTE_RUN,REMOTE_RUN+'/weights'):
    ensure_remote_dir(uri)

if not INIT_WEIGHT.is_file():
    print('Downloading initial seed85 best.pt ...')
    camber('cp',SOURCE_WEIGHTS,INIT_WEIGHT)
assert INIT_WEIGHT.is_file() and INIT_WEIGHT.stat().st_size > 40_000_000, INIT_WEIGHT
print('Initial checkpoint:',INIT_WEIGHT,round(INIT_WEIGHT.stat().st_size/2**20,1),'MiB')


Camber authentication OK via WSL: /home/dotru/.camber/bin/camber
Initial checkpoint: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\weights\v7p1_seed85_curated_yolo26m_best.pt 42.0 MiB


## 3. Restore tiến trình nếu máy từng bị ngắt
Nếu local đã có `last.pt`, notebook ưu tiên local. Nếu chưa có, nó thử tải checkpoint gần nhất từ Camber.

In [4]:
def try_restore(remote, local):
    if local.exists():
        return True
    local.parent.mkdir(parents=True, exist_ok=True)
    p = camber('cp', remote, local, check=False)
    if p.returncode == 0 and local.exists():
        print('[RESTORED]', local)
        return True
    return False

last_pt = RUN_DIR / 'weights/last.pt'
if not last_pt.exists():
    try_restore(REMOTE_RUN + '/weights/last.pt', last_pt)
    try_restore(REMOTE_RUN + '/results.csv', RUN_DIR / 'results.csv')
    try_restore(REMOTE_RUN + '/args.yaml', RUN_DIR / 'args.yaml')
print('Resume checkpoint:', last_pt if last_pt.exists() else 'không có — bắt đầu fine-tune mới')

Resume checkpoint: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p1_171126\yolo26m_finetune_v2_reviewed_181126\weights\last.pt


## 4. Train / resume
`last.pt` được ghi mỗi epoch. Cứ 5 epoch sẽ upload `last.pt`, `best.pt`, `results.csv`, `args.yaml` và checkpoint `epochN.pt` lên Camber.

In [5]:
def sync_file(local: Path, remote: str):
    if local.is_file():
        camber('cp', local, remote)

def sync_progress(epoch=None, final=False):
    ensure_remote_dir(REMOTE_RUN); ensure_remote_dir(REMOTE_RUN + '/weights')
    for rel in ('results.csv', 'args.yaml'):
        sync_file(RUN_DIR / rel, REMOTE_RUN + '/' + rel)
    for rel in ('weights/last.pt', 'weights/best.pt'):
        sync_file(RUN_DIR / rel, REMOTE_RUN + '/' + rel)
    if epoch:
        rel = f'weights/epoch{epoch}.pt'
        sync_file(RUN_DIR / rel, REMOTE_RUN + '/' + rel)
    status = {
        'experiment': EXPERIMENT, 'run': RUN_NAME, 'epoch': epoch, 'target_epochs': EPOCHS,
        'final_sync': final, 'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'dataset': str(DATA_YAML), 'source_weights': SOURCE_WEIGHTS,
    }
    status_path = RUN_DIR / 'camber_status.json'
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
    sync_file(status_path, REMOTE_RUN + '/camber_status.json')
    print(f'[CAMBER SYNC] epoch={epoch} -> {REMOTE_RUN}')

resume = last_pt.is_file()
import gc
gc.collect()
torch.cuda.empty_cache()
model = YOLO(str(last_pt if resume else INIT_WEIGHT))

def on_fit_epoch_end(trainer):
    # Ultralytics đã chạy validation ở cuối mỗi epoch; in một dòng gọn để dễ theo dõi.
    m = trainer.metrics or {}
    def value(key):
        v = m.get(key, float('nan'))
        try: return f'{float(v):.4f}'
        except Exception: return str(v)
    elapsed = (time.time() - train_clock['start']) / 60
    print(f'\n[V7.1 PROGRESS] epoch {trainer.epoch + 1}/{trainer.epochs} | '
          f'val mAP50={value("metrics/mAP50(B)")} | '
          f'val mAP50-95={value("metrics/mAP50-95(B)")} | '
          f'P={value("metrics/precision(B)")} R={value("metrics/recall(B)")} | '
          f'{elapsed:.1f} min elapsed', flush=True)

def on_model_save(trainer):
    epoch = trainer.epoch + 1
    if epoch % SYNC_EVERY == 0 or epoch == trainer.epochs:
        sync_progress(epoch=epoch, final=(epoch == trainer.epochs))

train_clock = {'start': time.time()}
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)
model.add_callback('on_model_save', on_model_save)

if resume:
    print('RESUME:', last_pt)
    results = model.train(resume=True, data=str(DATA_YAML.resolve()))
else:
    print('START fine-tune từ:', INIT_WEIGHT)
    results = model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
        optimizer='AdamW', lr0=1e-4, lrf=0.01, cos_lr=True, warmup_epochs=1.0,
        momentum=0.937, weight_decay=5e-4,
        mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0,
        translate=0.1, scale=0.5, degrees=0.0, shear=0.0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.4,
        amp=True, seed=SEED, deterministic=True, workers=2, cache=False,
        project=str(PROJECT), name=RUN_NAME, exist_ok=True,
        save=True, save_period=SYNC_EVERY, plots=True, patience=12, val=True, verbose=True,
    )
sync_progress(epoch=None, final=True)
print('Training finished:', RUN_DIR)

RESUME: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\runs\finding_v7p1_171126\yolo26m_finetune_v2_reviewed_181126\weights\last.pt
New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_human_reviewed_6cls_v2_171126\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, 

## 5. Xem best epoch và test cố định

In [6]:
import pandas as pd
csv_path = RUN_DIR / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
metric = 'metrics/mAP50-95(B)'
best_i = df[metric].idxmax()
display(df.loc[[best_i], ['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', metric]])

best = RUN_DIR / 'weights/best.pt'
assert best.is_file(), best
test_result = YOLO(str(best)).val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=0, plots=True)
print('Test mAP50   :', test_result.box.map50)
print('Test mAP50-95:', test_result.box.map)
sync_progress(epoch=None, final=True)

,epoch,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
19,20,0.86011,0.77843,0.83929,0.47057


Ultralytics 8.4.172  Python-3.12.2 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26m summary (fused): 130 layers, 20,354,078 parameters, 0 gradients, 68.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.40.0 ms, read: 12.211.7 MB/s, size: 99.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_human_reviewed_6cls_v2_171126\labels\test... 1931 images, 117 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1931/1931 361.1it/s 5.3s0.1s
val: New cache created: D:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\benchmakring_transmission_line_defect_model\data\finding_nemo_human_reviewed_6cls_v2_171126\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50 

### Lưu ý
- Không xóa thư mục `runs/finding_v7p1_171126/yolo26m_finetune` khi muốn resume.
- Cấu hình local đã kiểm chứng dùng `BATCH=8`, `workers=2`, `cache=False` trên RTX 4060 Laptop 8 GB/Windows WDDM. Sau mọi lỗi CUDA, restart kernel trước khi chạy lại.
- Kết quả test chỉ chạy sau khi đã chọn `best.pt` bằng val; không dùng test để chọn epoch.